# MELEK image pool — GPU runner (Kaggle / Colab)
Renders **background** image jobs from our pool on this notebook's free GPU, with the same pipeline our servers use, and posts them back. Stops by itself when idle.

**Before running:** turn on the GPU (Kaggle: Settings → Accelerator → GPU T4; Colab: Runtime → Change runtime type → T4) and add two secrets: `POOL_URL` and `PULL_TOKEN` (Kaggle: Add-ons → Secrets; Colab: the key icon). If the pool's lane is switched off, the runner exits right away.

In [ ]:
!pip -q install diffusers==0.40.0 transformers==5.17.0 accelerate==1.15.0 peft==0.21.0 safetensors opencv-python-headless pillow

In [ ]:
import os, urllib.request
RAW='https://raw.githubusercontent.com/HinduTempleCoins/Bot/main/integrations/'
os.makedirs('gpu', exist_ok=True)
for f, dst in [('genai_cpu_worker.py','genai_cpu_worker.py'), ('gpu/gpu_puller.py','gpu/gpu_puller.py')]:
    urllib.request.urlretrieve(RAW + f, dst)
print('fetched')

In [ ]:
def secret(k):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(k)
    except Exception:
        from google.colab import userdata
        return userdata.get(k)
os.environ.update(POOL_URL=secret('POOL_URL'), PULL_TOKEN=secret('PULL_TOKEN'), CPU_SD_DEVICE='cuda',
                  RUNNER_NAME=('kaggle' if os.path.exists('/kaggle') else 'colab'), MAX_MINUTES='540', IDLE_MINUTES='15')
print('runner', os.environ['RUNNER_NAME'])

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, 'gpu')
import gpu_puller
gpu_puller.POOL, gpu_puller.TOKEN = os.environ['POOL_URL'].rstrip('/'), os.environ['PULL_TOKEN']
print('images made:', gpu_puller.run())